## LLM Chess Showdown &mdash; Data Collection

Simulates chess games between two LLMs:
- OpenAIs pgt-4-turbo
- Anthropic's claude-sonnet-4-5-20250929

The AI's communicate using their respective APIs. Each AI is prompted to respond with UCI moves (e.g. e2e4) and an optional comment. Moves are validated using the chess library.

**Originally built in Google Colab.** This version was adapted to run locally &mdash; Colab-specific cells have been removed and hardcoded paths replaced with relative laths 'os.path.join' and API keys moved to environment variables (see '.env.example')

In [ ]:
import os
import time
import json
import re
import chess
import openai
import anthropic

from dotenv import load_dotenv

### Configuration & Constants

**Value configuration:** choose which model starts/plays (Black or White), where game data is written, and how many retries the models get per turn before their turn is forfeited for an illegal move.

**Working Directory:** the path below assumes the notebook runs from inside 'notebooks/', the default when opening the file directly. If your environment launches Jupyter from the repo root instead, you will get a FileNotFoundError. Check with 'print(os.getcwd())' to confirm, and change 'DATA_DIR' to 'os.path.join("data", ...)' if needed.

**Note:** if you update whick model plays the Black or White piece ('WHITE_AI_MODEL', 'BLACK_AI_MODEL') you will need to update the Rate Limit Handeling Section below as the models RPM/TPM rate limits are hardcoded at this time.

In [ ]:
# Constants
DATA_DIR = os.path.join("..", "data")
RAW_DATA_FILENAME = os.path.join(DATA_DIR, "raw_data.json")

# Hardcoded AI models for White and Black
WHITE_AI_MODEL = "gpt-4-turbo"
BLACK_AI_MODEL = "claude-sonnet-4-5-20250929"

# Max retries for an AI to make a legal move in a single turn
MAX_RETRIES_PER_TURN = 3

### API Setup

Loads API keys from environment variables via a local .env file (not commited to the repo &mdash; see '.env.example' for expected format). and initializes the OpenAI and Anthropic clients used for all move requests.

**Setup:** copy '.env.example' to '.env' in the repo root and fill in your own 'OPENAI_API_KEY' and 'ANTHROPIC_API_KEY' before running this cell.

In [ ]:

load_dotenv()

API_KEY_OPEN = os.getenv("OPENAI_API_KEY")
API_KEY_ANTH = os.getenv("ANTHROPIC_API_KEY")

# Initialize OpenAI and Anthropic clients
client_open = openai.Client(api_key=API_KEY_OPEN)
client_anth = anthropic.Client(api_key=API_KEY_ANTH)

### Rate Limit Handling
OpenAI and Anthropic enforce rate limits differently, so each is tracked separately here rather than with shared logic:

- **OpenAI:** &mdash; limited by _requests-per-minute_ and total _tokens-per-minute_, tracked with global counters that reset every 60 seconds.
- **Anthropic:** &mdash; limited by _requests-per-minute_, but tracks _input_ and _output_ tokens separately, each against its own cap.

**Global Variables:** As seen in cell 8, these variables are tracked separately and initialize that tracking state at 0/"now" before any API call happens. They are declared here rather than inside the functions because they read and update via _global_ on every call. They need to persist accross the whole session for rate limits to actually work.

**Note:** The RPM/TPM values below are hardcoded for the specific models set in the Constants cell 
- gpt-4-turbo,
- claude-sonnet-4-5-20250929
If you change either model, these limits need to be updated to reflect your API plan and the chosen models rate limits

In [ ]:
# ------------------ Rate Limit Handling ------------------
# Global variables to track rate limits for OpenAI
openai_requests_made = 0
openai_tokens_used = 0
openai_last_reset_time = time.time()
openai_rate_limit_hits = 0

# Global variables to track rate limits for Anthropic
anthropic_requests_made = 0
anthropic_input_tokens_used = 0
anthropic_output_tokens_used = 0
anthropic_last_reset_time = time.time()
anthropic_rate_limit_hits = 0

In [ ]:
def handle_openai_rate_limits(max_tokens_per_request):
    global openai_requests_made, openai_tokens_used, openai_last_reset_time, openai_rate_limit_hits

    while True:
        current_time = time.time()
        # Check if we need to reset the counters (start of a new minute)
        if current_time - openai_last_reset_time >= 60:
            openai_requests_made = 0
            openai_tokens_used = 0
            openai_last_reset_time = current_time

        # Check if request limit has been reached (OpenAI)
        if openai_requests_made >= 500: # 500 RPM for gpt-4-turbo
            print("OpenAI Request limit reached, waiting until next minute...")
            time.sleep(60 - (current_time - openai_last_reset_time))
            continue

        # Check if token limit has been reached (OpenAI)
        if openai_tokens_used + max_tokens_per_request > 30000: # 30,000 TPM
            print("OpenAI Token limit reached for this minute, waiting until next minute...")
            time.sleep(60 - (current_time - openai_last_reset_time))
            continue
        break # Limits are fine, proceed


def make_api_request(messages, model, max_tokens_per_request=1000):
    global openai_requests_made, openai_tokens_used, openai_rate_limit_hits
    global anthropic_requests_made, anthropic_input_tokens_used, anthropic_output_tokens_used, anthropic_last_reset_time, anthropic_rate_limit_hits

    while True:
        try:
            # Request to the AI API (OpenAI or Anthropic)
            if model.startswith('gpt'):  # OpenAI models
                handle_openai_rate_limits(max_tokens_per_request)
                response = client_open.chat.completions.create(
                    model=model,
                    messages=messages,
                    max_tokens=max_tokens_per_request
                )
                # Update OpenAI request and token counts
                openai_requests_made += 1
                openai_tokens_used += response.usage.total_tokens

                return response

            else:  # Anthropic models
                current_time = time.time()
                if current_time - anthropic_last_reset_time >= 60:
                    anthropic_requests_made = 0
                    anthropic_input_tokens_used = 0
                    anthropic_output_tokens_used = 0
                    anthropic_last_reset_time = current_time

                system_message = None
                # Anthropic expects the system message as a separate parameter
                if messages and messages[0].get('role') == 'system': # Removed extra parenthesis
                    system_message = messages[0].get('content')
                    # Remove system message from the messages list for Anthropic API call
                    anthropic_messages_for_api = messages[1:]
                else:
                    anthropic_messages_for_api = messages

                # Anthropic RPM check (50 RPM for claude-sonnet-4-5-20250929)
                if anthropic_requests_made >= 50:
                    print("Anthropic Request limit reached, waiting until next minute...")
                    time.sleep(60 - (current_time - anthropic_last_reset_time))
                    continue

                # Anthropic Output Token limit check (8,000 Output TPM for claude-sonnet-4-5-20250929)
                if anthropic_output_tokens_used + max_tokens_per_request > 8000: # requested output limit
                    print("Anthropic Output Token limit reached for this minute, waiting until next minute...")
                    time.sleep(60 - (current_time - anthropic_last_reset_time))
                    continue

                response = client_anth.messages.create(
                    model=model,
                    max_tokens=max_tokens_per_request,
                    messages=anthropic_messages_for_api, # Pass processed messages to API
                    system=system_message  # Pass system message here
                )
                # Update Anthropic request and token counts
                anthropic_requests_made += 1
                anthropic_input_tokens_used += response.usage.input_tokens # Use actual input tokens
                anthropic_output_tokens_used += response.usage.output_tokens # Use actual output tokens

                if anthropic_input_tokens_used > 30000:
                    print("Anthropic Input Token limit *exceeded* for this minute after request, waiting until next minute...")
                    time.sleep(60 - (current_time - anthropic_last_reset_time))
                    continue

                return response
        except openai.APIStatusError as e:
            if e.status_code == 429: # Rate limit hit
                openai_rate_limit_hits += 1 # Renamed for consistency
                print(f"GPT Rate limit hit {openai_rate_limit_hits} time(s). Waiting 60 seconds...")
                time.sleep(60) # Wait before retrying
                continue
            else:
                print(f"OpenAI API Error: {e}")
                time.sleep(60) # Wait before retrying
                continue
        except anthropic.APIStatusError as e:
            if e.status_code == 429: # Rate limit hit
                anthropic_rate_limit_hits += 1 # Increment Anthropic rate limit counter
                print(f"Anthropic Rate limit hit {anthropic_rate_limit_hits} time(s). Waiting 60 seconds...")
                time.sleep(60) # Wait before retrying
                continue
            else:
                print(f"Anthropic API Error: {e}")
                time.sleep(5) # Wait before retrying
                continue
        except Exception as e:
            print(f"Error making API request: {e}")
            time.sleep(5)  # Wait before retrying
            continue  # Retry the request

### AI Move Generation

Requests a move from whichever AI is currently in play, constructing a system prompt that identifies the AI's color and instructs it to make a single UCI move, followed by the option for a one line comment. If the AIs move was illegal, according to UCI moves, that feedback is appended to the prompt so the AI can correct itself.

In [ ]:
def get_ai_move(board, last_move, ai_model, feedback=None, game_feedback_prefix=""):
    # Determine the AI's color based on whose turn it is
    ai_color = "White" if board.turn == chess.WHITE else "Black"

    # Construct the base system message
    system_message_content = (
        game_feedback_prefix + # Prepend game outcome feedback
        f"You are a chess engine playing as {ai_color}. "
        "Your response must begin with a single, valid, and legal UCI chess move for "
        f"{ai_color} on the current board (e.g., 'e2e4'). "
        "Optionally, you may follow the move with a brief comment about your move"
        "on a new line, starting with 'Comment: '. "
        "Example output: 'e2e4\nComment: I'm deploying my king's pawn.'"
        "DO NOT share your stategy."
    )

    # Add feedback to the user message if available
    user_message_content = f"Current board:\n{board}\nLast move: {last_move if last_move else 'None'}\nOutput your move (e2e4) and optional comment:"
    if feedback:
        user_message_content += f"\n\nPrevious attempt was illegal: {feedback}. Please provide a new, legal move and optional comment."

    messages = [
        {"role": "system", "content": system_message_content},
        {"role": "user", "content": user_message_content}
    ]

    response = make_api_request(messages, ai_model)

    if ai_model.startswith('gpt'):
        full_output = response.choices[0].message.content.strip()
    else: # Anthropic models
        full_output = response.content[0].text.strip()

    move = ""
    comment = ""

    # Split the output into lines
    output_lines = full_output.split('\n')

    # Attempt to extract a valid UCI move from the first line
    uci_pattern = r"[a-h][1-8][a-h][1-8]([qrbn])?"
    match = re.search(uci_pattern, output_lines[0])

    if match:
        move = match.group(0).strip()
        # Check for a comment in subsequent lines
        for line in output_lines[1:]:
            if line.strip().startswith("Comment:"):
                comment = line.strip().replace("Comment:", "", 1).strip()
                break # Only take the first comment line
    else:
        # Fallback if no UCI move found on the first line. Treat the whole output as potentially just a move (if it matches UCI pattern)B
        move = output_lines[0].strip()
        # Still try to extract comment if move was not found by main regex
        for line in output_lines[1:]:
            if line.strip().startswith("Comment:"):
                comment = line.strip().replace("Comment:", "", 1).strip()
                break

    return move, comment, messages # Return move, comment, and messages

### Save Session Data

Writes all games played in a session to 'raw_data.json'. If the file already exists, its loaded first so new games are appened to the new game session to the games list in the 'raw_data.json' file. It also will store the initial system message one so they aren't duplicated on every save.

In [ ]:
def save_session_data(all_game_datas, session_initial_messages):
    # Ensure the directory exists before attempting to create the file
    os.makedirs(os.path.dirname(RAW_DATA_FILENAME), exist_ok=True)

    master = {}
    if os.path.exists(RAW_DATA_FILENAME):
        try:
            with open(RAW_DATA_FILENAME, "r") as f:
                master = json.load(f)
        except json.JSONDecodeError:
            print(f"Warning: {RAW_DATA_FILENAME} is corrupted or empty. Starting new file.")
            master = {}

    if "games" not in master:
        master["games"] = []

    # Add session initial messages if not already present in the file and provided
    if session_initial_messages and "session_initial_messages" not in master:
        master["session_initial_messages"] = session_initial_messages

    # Append new game data
    for game_data in all_game_datas:
        master["games"].append(game_data)

    with open(RAW_DATA_FILENAME, "w") as f:
        json.dump(master, f, indent=4)

    print(f"All session data saved to {RAW_DATA_FILENAME}")

### Game Loop

Plays a single game of chess between the two AI models start to finish:
- Requiests move in turn,
- validates turn,
- tracks timing and move history, and
- determines the game's outcome (win, loss, draw, or forfeit after repeated illegal moves)

Returns the completed game's data for saving

In [ ]:
def play_game(game_number, white_ai_model, black_ai_model, total_session_time_start, white_game_start_message="", black_game_start_message=""):
    # Play one game between the two AI models
    board = chess.Board()
    turn = "white"
    last_move = None
    MAX_MOVES = 300
    move_number = 1
    white_time = 0
    black_time = 0
    detailed_moves = []
    game_result = None # Initialize game_result to None
    failed_player_color = None # To store which player failed to make a move
    failed_player_move = None # To store the attempted illegal move

    # Construct the full initial system messages for White and Black for this game (including previous game feedback)
    white_initial_system_message_for_this_game = white_game_start_message + (
        f"You are a chess engine playing as White. "
        "Your response must begin with a single, valid, and legal UCI chess move for "
        f"White on the current board (e.g., 'e2e4'). "
        "Optionally, you may follow the move with a brief comment about your move"
        "on a new line, starting with 'Comment: '. "
        "Example output: 'e2e4\nComment: I'm deploying my king's pawn.'"
        "DO NOT share your stategy."
    )
    black_initial_system_message_for_this_game = black_game_start_message + (
        f"You are a chess engine playing as Black. "
        "Your response must begin with a single, valid, and legal UCI chess move for "
        f"Black on the current board (e.g., 'e2e4'). "
        "Optionally, you may follow the move with a brief comment about your move"
        "on a new line, starting with 'Comment: '. "
        "Example output: 'e2e4\nComment: I'm deploying my king's pawn.'"
        "DO NOT share your stategy."
    )

    if game_number == 1:
        print(f"\nWhite Initial System Message for Game {game_number}:\n{white_initial_system_message_for_this_game}")
        print(f"\nBlack Initial System Message for Game {game_number}:\n{black_initial_system_message_for_this_game}")

    print(f"\n=== GAME {game_number} ===")
    print(f"White: {white_ai_model}")
    print(f"Black: {black_ai_model}")

    game_start_time = time.time() # Start timer for the current game

    # Flags to ensure game start messages are only applied once per AI per game
    white_initial_message_used = False
    black_initial_message_used = False

    while not board.is_game_over() and move_number <= MAX_MOVES and game_result is None:
        ai_model = white_ai_model if board.turn == chess.WHITE else black_ai_model
        current_player_color = "White" if board.turn == chess.WHITE else "Black"

        current_game_start_prefix = ""
        # Apply game start message only for the first move of each player in this game
        # This current_game_start_prefix will carry the 'feedback' part (if any) to get_ai_move
        if board.turn == chess.WHITE and not white_initial_message_used:
            current_game_start_prefix = white_game_start_message
            white_initial_message_used = True
        elif board.turn == chess.BLACK and not black_initial_message_used:
            current_game_start_prefix = black_game_start_message
            black_initial_message_used = True

        move_attempts = 0
        uci_move = None
        move_comment = "" # Initialize variable for comment
        move_obj = None
        feedback_to_ai = None
        ai_messages_for_turn = [] # List to store all messages for this turn
        move_start_time = time.time() # Start timer for the whole turn

        while move_attempts < MAX_RETRIES_PER_TURN:
            # Pass current_game_start_prefix only for the first attempt of the turn, then clear it.
            prefix_for_this_attempt = current_game_start_prefix if move_attempts == 0 else ""

            current_uci_move, current_move_comment, current_messages = get_ai_move(board, last_move, ai_model, feedback=feedback_to_ai, game_feedback_prefix=prefix_for_this_attempt)
            ai_messages_for_turn.append(current_messages) # Store messages for this attempt
            uci_move = current_uci_move
            move_comment = current_move_comment # Store the comment

            try:
                move_obj = chess.Move.from_uci(uci_move)
                if move_obj not in board.legal_moves:
                    feedback_to_ai = f"Move '{uci_move}' is not a legal move for {current_player_color} on the current board."
                    print(f"Attempt {move_attempts + 1}: Illegal move received from {ai_model}: {uci_move!r} (Error: {feedback_to_ai})")
                    move_attempts += 1
                    continue
                break # Valid and legal move found
            except chess.InvalidMoveError as e:
                feedback_to_ai = f"Move '{uci_move}' is not a valid UCI format or is not pseudo-legal: {e}"
                print(f"Attempt {move_attempts + 1}: Invalid UCI format or pseudo-legal error from {ai_model}: {uci_move!r} (Error: {e})")
                move_attempts += 1
                continue
            except Exception as e:
                feedback_to_ai = f"An unexpected error occurred: {e}"
                print(f"Attempt {move_attempts + 1}: Unexpected error from {ai_model}: {uci_move!r} (Error: {e})")
                move_attempts += 1
                continue

        # If no legal move after retries, set game_result and break game loop
        if move_attempts == MAX_RETRIES_PER_TURN:
            print(f"Player {ai_model} ({current_player_color}) failed to make a legal move after {MAX_RETRIES_PER_TURN} attempts. Forfeiting turn.")
            failed_player_color = current_player_color
            failed_player_move = uci_move if uci_move else "No valid move generated"
            if current_player_color == "White":
                game_result = "0-1" # White failed, Black wins
            else:
                game_result = "1-0" # Black failed, White wins
            break # Exit game loop

        # If we reached here, a legal move was successfully made
        board.push(move_obj);
        last_move = uci_move
        print(board) # Print the ASCII board after each move
        if move_comment: # Print comment if available
            print(f"{current_player_color} AI Comment: {move_comment}")

        # Time tracking for AI move (total time for this turn)
        move_t = time.time() - move_start_time
        if turn == "white":
            white_time += move_t
        else:
            black_time += move_t

        # Track move details
        detailed_moves.append({
            "number": move_number,
            "player": turn,
            "ai": ai_model,
            "move": uci_move,
            "comment": move_comment, # Add comment here
            "time": round(move_t, 2),
            "ai_messages_for_turn": ai_messages_for_turn, # Store messages for the successful turn
            "board_state_ascii": str(board) # Add ASCII representation of the board after the move
        })

        # Update turn
        turn = "white" if turn == "black" else "black"
        move_number += 1

    # Determine the final result if not already set by illegal move
    if game_result is None:
        game_result = board.result()

    # Determine the winning AI for logging
    winner_ai = ""
    if game_result == "1-0":
        winner_ai = white_ai_model
    elif game_result == "0-1":
        winner_ai = black_ai_model
    elif game_result == "1/2-1/2":
        winner_ai = "Draw"
    else:
        winner_ai = "Inconclusive"

    # Calculate total game time
    total_game_time = round(time.time() - game_start_time, 2)

    # Prepare the game data to save to the JSON file
    game_data = {
        "game_number": game_number,
        "white_initial_system_message": white_initial_system_message_for_this_game, # Store initial system message for White
        "black_initial_system_message": black_initial_system_message_for_this_game,  # Store initial system message for Black
        "white_ai": white_ai_model,
        "black_ai": black_ai_model,
        "moves": detailed_moves,
        "white_total_time": round(white_time, 2),
        "black_total_time": round(black_time, 2),
        "total_game_time": total_game_time, # Added total game time
        "result": game_result,
        "winner_ai": winner_ai, # Explicitly state the winning AI
        "failed_player_color": failed_player_color, # Store if a player failed to make a legal move
        "failed_player_move": failed_player_move, # Store the attempted illegal move
        "openai_rate_limit_hits": openai_rate_limit_hits,
        "anthropic_rate_limit_hits": anthropic_rate_limit_hits,
        "total_session_time_start": total_session_time_start # Pass this along to game data

    }

    return game_data, white_initial_system_message_for_this_game, black_initial_system_message_for_this_game # Return game_data and initial messages


### API Connection Test

Sends a test request to both OpenAI and Anthropic APIs to confirm the keys and clients are in working order before any games are played. Runs once at the start of 'main()'.

**Known Inconsistency:** this function hardcodes "gpt-4-turbo' and "claude-sonnet-4-5" directly, separetely from the 'WHITE_AI_MODEL"/"BLACK_AI_MODEL" constants set above. So, the constants are changed, test_api_connections() must be updated as well, and vice versa.

In [ ]:
def test_api_connections():
    # ChatGPT (OpenAI) API connection test
    try:
        print("Connecting to OpenAI API...")
        response = client_open.chat.completions.create(
            model="gpt-4-turbo",  # Use an appropriate model version (e.g., gpt-3.5, gpt-4)
            messages=[{"role": "user", "content": "Say Hello"}],
            max_tokens=10
        )
        print("Connected to OpenAI API!!")
    except Exception as e:
        print(f"Error connecting to OpenAI API: {e}")

    # Anthropic API connection test
    try:
        print("Connecting to Anthropic API...")
        message = client_anth.messages.create(
            model="claude-sonnet-4-5",
            max_tokens=1000,
            messages=[{"role": "user", "content": "Hello"}]
        )
        print("Connected to Anthropic API!!")
    except Exception as e:
        print(f"Error connecting to Anthropic API: {e}")

### Main
The entire point of this notebook:
- Tests API connections, 
- prompts the user for the number of games, 
- prompts user for position swaping (winner plays next game first), 
- track previous games winner, looser, illegal moves to determin which AI starts the next game, and
- saves all session data once every game is completed.

In [ ]:
def main():
    # Test API connections before starting the game
    test_api_connections()

    total_games = int(input("How many games do you want to play? "))

    # Ask the user if they want to swap AI models after each game
    swap_choice = input("Do you want to swap AI models (White and Black) after each game? (yes/no): ").strip().lower()

    # Hardcoded initial models
    white_ai_model = WHITE_AI_MODEL
    black_ai_model = BLACK_AI_MODEL

    # Store feedback messages for each AI model, to be passed to the next game
    model_feedback_messages = {
        white_ai_model: "",
        black_ai_model: ""
    }

    total_session_time_start = time.time() # Start timer for the entire session
    all_game_datas = [] # To store all game data before final save
    session_initial_messages = {} # To store initial system messages once for the session

    game_number = 1
    while game_number <= total_games:
        current_white_player_model = white_ai_model
        current_black_player_model = black_ai_model

        # Get feedback messages for the models playing in this game
        white_game_start_message = model_feedback_messages[current_white_player_model]
        black_game_start_message = model_feedback_messages[current_black_player_model]

        game_data_current, white_init_msg_current, black_init_msg_current = play_game(game_number, current_white_player_model, current_black_player_model, total_session_time_start,
                                                                                       white_game_start_message, black_game_start_message)

        all_game_datas.append(game_data_current)

        # Capture initial system messages for session-level saving if it's the first game
        if game_number == 1:
            session_initial_messages = {
                "white_initial_system_message": white_init_msg_current,
                "black_initial_system_message": black_init_msg_current
            }

        # Clear previous feedback messages as they've been delivered for this game's start
        model_feedback_messages[current_white_player_model] = ""
        model_feedback_messages[current_black_player_model] = ""

        result = game_data_current['result'] # Get result from the returned game_data
        failed_player_color = game_data_current['failed_player_color']
        failed_player_move = game_data_current['failed_player_move']

        # Construct feedback for the next game based on the current game's result
        if result == "1-0":  # White wins
            print(f"Winner: {current_white_player_model} (White)")
            model_feedback_messages[current_white_player_model] = "Your last game ended in a victory! "
            if failed_player_color == 'Black':
                model_feedback_messages[current_black_player_model] = f"You lost your last game because you made an illegal move ({failed_player_move}). "
            else:
                model_feedback_messages[current_black_player_model] = "You lost your last game. "
        elif result == "0-1":  # Black wins
            print(f"Winner: {current_black_player_model} (Black)")
            model_feedback_messages[current_black_player_model] = "Your last game ended in a victory! "
            if failed_player_color == 'White':
                model_feedback_messages[current_white_player_model] = f"You lost your last game because you made an illegal move ({failed_player_move}). "
            else:
                model_feedback_messages[current_white_player_model] = "You lost your last game. "
        elif result == "1/2-1/2": # Draw
            print("Game was a draw.")
            model_feedback_messages[current_white_player_model] = "Your last game was a draw. "
            model_feedback_messages[current_black_player_model] = "Your last game was a draw. "
        else: # Game ended inconclusively (e.g., max moves reached without checkmate/stalemate/illegal move)
            print("Game ended inconclusively.")
            model_feedback_messages[current_white_player_model] = "Your last game ended inconclusively. "
            model_feedback_messages[current_black_player_model] = "Your last game ended inconclusively. "

        # If the user wants to swap models, update the model names for the *next* game
        if swap_choice == "yes":
            # If Black won or it was a draw, swap positions for the next game
            # Note: We swap the *names* which ensures the correct feedback follows them in the model_feedback_messages dict.
            if result == "0-1" or result == "1/2-1/2":
                white_ai_model, black_ai_model = black_ai_model, white_ai_model  # Swap positions for next game
                print(f"Models swapped for the next game. White: {white_ai_model}, Black: {black_ai_model}")

        game_number += 1

    total_session_time_end = time.time()
    total_session_duration = round(total_session_time_end - total_session_time_start, 2)
    print(f"\nAll {total_games} games finished in {total_session_duration} seconds.")

    # Save all game data and session initial messages to the master JSON file once
    save_session_data(all_game_datas, session_initial_messages)

In [ ]:
if __name__ == "__main__":
    main()